# Tiny-GenImage JPEG-bias evaluation — CLIP, NPR-ResNet18, Full AIDE

Notebook này **không train lại**. Nó giữ nguyên ba checkpoint đã train trên Tiny-GenImage và kiểm tra ảnh test theo các trường hợp:

1. `raw` — baseline bắt buộc;
2. `png_roundtrip` — decode → PNG → decode;
3. `fake_jpeg96` — giữ real gốc, chỉ nén fake JPEG quality 96;
4. `all_jpeg96` — nén cả real và fake JPEG quality 96;
5. `controlled_jpeg96` — real gốc có estimated quality 94–98 và fake lossless được nén JPEG96;
6. `raw_on_controlled_cohort` — baseline cùng membership với controlled cohort.

Notebook audit số ảnh, format, kích thước gốc, aspect ratio, megapixel, dung lượng file và JPEG quality trước khi inference. Mọi biến đổi codec diễn ra **trước preprocessing gốc của từng model** và không ghi đè Tiny-GenImage.

## Kaggle cần chuẩn bị

- GPU accelerator;
- Add Input `yangsangtai/tiny-genimage`;
- Add Input là Kaggle Dataset tạo từ folder `checkpoints/` đi kèm notebook;
- bật Internet để tải CLIP ViT-B/32 và OpenCLIP ConvNeXt-XXLarge, hoặc attach `open_clip_pytorch_model.bin`;
- tùy chọn: Kaggle Secret `HF_TOKEN`.

Full AIDE là **AIDE original Full**, không phải forensic-only và không phải AIDE–NPR.

In [ ]:
%pip install -q "open_clip_torch==2.26.1" "pandas<3.0" "scikit-learn<1.9" "pillow<12.0" "tqdm"

In [ ]:
import os
from pathlib import Path

import torch

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    print("HF_TOKEN is not configured; public downloads will be anonymous.")

assert torch.cuda.is_available(), "Kaggle Settings → Accelerator → GPU trước khi chạy."
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

In [ ]:
CONFIG = {
    "input_root": "/kaggle/input",
    "dataset_root": None,  # None = tự tìm Tiny-GenImage.
    "output_root": "/kaggle/working/tiny_jpeg_bias_three_models_results",

    # Full run: 500 real + 500 fake mỗi generator. Đổi smoke=True để thử 100+100.
    "smoke": False,
    "max_per_class_per_generator": None,
    "selection_seed": 42,

    "models": ["clip_linear_probe", "npr_resnet18", "aide_original_full"],
    "cases": [
        "raw",
        "png_roundtrip",
        "fake_jpeg96",
        "all_jpeg96",
        "raw_on_controlled_cohort",
        "controlled_jpeg96",
    ],

    "jpeg_quality": 96,
    "jpeg_subsampling": 0,
    "jpeg_optimize": False,
    "jpeg_progressive": False,
    "controlled_real_quality_min": 94,
    "controlled_real_quality_max": 98,
    "controlled_balance_across_generators": True,

    "clip_batch_size": 32,
    "npr_batch_size": 32,
    "aide_batch_size": 4,
    "num_workers": 2,
    "threshold": 0.5,

    # Notebook lưu partial CSV. Chạy lại cell sẽ bỏ qua sample đã hoàn thành.
    "resume": True,
    "save_every_batches": 25,

    # None = tự tìm chính xác các filename trong /kaggle/input và xác minh SHA256.
    "clip_checkpoint": None,
    "npr_checkpoint": None,
    "aide_checkpoint": None,
    "verify_checkpoint_hashes": True,

    # None = tự tìm open_clip_pytorch_model.bin; nếu không có sẽ tải từ HF.
    "aide_semantic_checkpoint": None,
    "clip_pretrained": "openai",
}

CONFIG

In [ ]:
import base64
import gzip
import importlib.util
import sys
from pathlib import Path

RUNNER_GZIP_BASE64 = """H4sIAAAAAAACCu19f3PbSI7o//4UHL5676iswsiK7di+p63yOs5s7jJOLsncbpVOxaUkyuZFIjUildiT8/vsD0D/QjeblJzJ3VzV7VRNLJLdaDQajQbQaHQYhlef0+U2rbOgvt1kWfAxL+6f/pgVr1fpTRbMbrPZp3WZF3UVbIt5tglmZVFvyuUymwc5Fnk6K+fZLKhu80VdxQcHP8Hjsjp4Sv8dPAku37x+F/xr/vHpn549HwZf8voWGoLGNmleAIwl/JtugvWmnGYxFL9+9/7p+6y6zurDU11osSlXQTXbpPXsFgu93eQ3eZEug1fb5TK4eP3yKojerrOC2qqyVVrU+SyYbtJidhv8IXh5+fHZh/c/BYtykxWV/tIDbD8CKpstFF9lQbqd59BNxK5I6/yzSwugUz6HD2URLPJlVvWD2SYDwlUAeLs52GS/bLOqBnQB7aKCxlZUuAryIlhlq3Jz3w/S+bwKNumXYJMtMsBmBiQub8tNDdAyORBVkC6XByuioyZYvglK1ev1JgN6zbKqyosbgFnMgy+bHCtusmq7SqfLDMvM8xm13z9YZfUmn0ET62WJLWENgPA5K1LEoC6Dv/3t2af05maZPftSbj4B2L/9DahzXZq+qPEOoNpGNoekqqDvAAMIk1ZZHQf/9O7qx2fvrn8MNiUwzFNoeF0Ft+kahudgmuEQBFkKw0L9+4fK9Apwz3/Nns025fpZgcRb5r8Kaq/zdYZ8Eh+EYXhwQNyQJIttvd1kSQJorYGA0KmirAXFDw7ku5uZ+nWbVrfLfKoexR94EW/rfKnflurXv1dloX7DMN6q32Wlfq2XaY10Uc9Ap3mpn6p7XRB5S//epLNsms4+iU6sATLgoHrwDhuiD/X9GsZAvb8ogHMugSlwZPvB6zrb4K8DhiAOLHbvwPyOt1UWhRc3N2GvWTBe3+OvIK2gI7X6XmxX63t8V6x1N6Fb8ALLzdW7d6/f6A6Vm9mt9RAXBUEo3LfxYlsQQ8JYQ4FXoqcAS3WTZllf/HkFE0wUqD4tQUAUseRhVTg6COC/dDbbAknvk2oGjNUX74A/AUICM2CWV9Ae/zhNl8jy88RXE0TbYks1gFCb/E68XRzyMl6o8A5Gx3pTzpJ0O1OvenJYiRTIcVWME0Z15iX8flOmIF779BsmEqvwmdoLDAuJCVnJIr/MV3G6hTmsCsCLg4MDTcb4zduLl8nH9z9fX158vHqZvP7p4serD8Eo+LjZAg/RY3L1149X1x9ev73GD1/D+N/XN2E/wL+Z+LEuxN/pak1/v2RT8aPOF+rvInw4eHPxp6s3ycvX768uP759/5oaiqIQBCpMVig4gLIgNZdhrx9EYZrDq0N4tUg/ZWGvd/D+6l9+vvqAaF5efBCViZ4htJ+QREGBEgoiU60EUTw7Ua9wHOw3ZsEyH7ChV1fvr64vr0xDIchl7Ar8SWB8WT0homEeXbx5o8u7EP4QOMgfXP31HVABHi/eX/759Uf4/fP7K9MlWrcWuILd3k83+fw80GvYZVl8vs7+Wj/961/fpBsQumYRm29h/ogV8ngQQk9MM5d/vrr853dvX19/TD78+WJ4fIJjKYmwzNeJWGsTWmvD8yB8cXp6dHJ4NDycTY9Pj6fTxdHpWZpOZy9OFs+Pn78YHJ/O5i9eHD/PZrPF2dHz4fx5+uJseDIcLl48Px0cv1AULtabBKR3gWs2wn0+PF2cLY6eH86fv4Bfx8PT2enJ2cnJi7OT7GzxPJ2dDI7S41k2nC6ez58PXgyGp4OT4YvByewQml0M9Fjm8yxR60OClELoJ8PTo9PTs+Pp0dHw+Hh+kg6OTwDLo9nx4Dn8Ph28yOaDdHg2PR0cQUdmJ2fHUObo7PToKDt6MXsO0B8OGKlevX5zdX3xE41pF7X4y9ssncfruo0A/DnBWZoo5cXU8Xeu+TYhJQjlvaj8cHDw8urVxc9vPiaXb69fvf7xPMCFflzVIDxgpZiYbuTFelvDrClrBK0WeHqrsJDLtip0XRZSgIXltvbVlurBsxpUI5pPyTRPq4R0x0ToLNj17bKuVBtGb0qq9TIncPBOfa6yZTYTX7NsDh+PhurLqvyEpH+VLiuF1iq9S9bZJpkt06qiXzdZAQIfJKWNv8AF3o09w9l3xqzvHY+JkiBAIoS0zKs60iKgJ79i16vbcomoD+Jj+ZZI8wvM1by+hw9nJ/x9tZ1W6QpoAUIVKvFP5Rr0BdCDnG7TN8D9ZoNa32f3M5NVKFxVw8kqL7Dxo13F0jssdtosJtdLWC43JdBb0xrJgetHn02YKXJ4Ugnknw/Z3PB/IYpbnxSeoIckyGgZtaPZAVb2JIPV/V7UojEZKoKT6pvZaEHZfHGfGCMmQTUw8yFvythshOi3faMOtH0kqKAtSBsGmb4E8Y4rHqutjJV2MOVqBVND2igJaoweJLyfQVIcfPh4cf3y4v3L5M3PP72+vsDV6l8+XvzpDS5DxTpOq3SzSe/FgjSmf/G/wxNYlWFZPoTVGn8Pj/rBEfw+hncnh31TbtgX/8P3wzMoB2WPT6EMlj1m5fD7cwfWCygHdY5PnHLwfggwh2eivVN4PoXyJ0NW7lSUef6C6gcnp4jrGf4Drbx4YUpia8+PCZvgBH6fUq+wGcTnjME8OhMlXpyKNgnU4RDLD5EMA9bvF0OqG5wB1DNsm4gwGMhaZ2eiqJQfc9DmsxFQe7Es0xrawEX7YJ4tyMpI4EcK4jJC6y87RxHeC57+Ef+eU/V8EYACWFQ1TkRRqk+mQu9cIwRstt0UAawBokCvvWYEiACXZTegbPaaIODTXiCoL2h+emDQt51QAAiYFsh+TQhUJK5LkrgCBkyjCixyIOXVZlNuokV4SRZf8E8f3l4/rWCmo8GYBV+R2rLxHzYPoaI1SQ8keIST5JwoCAZxeg/Izg3ZceoIdLBYvE7BSK/j1ad5vonEQzUi2RFkd4BcUn6iR4FjnaEKnm7uYXZRdbTeQdovFvkdNRuL36DOgcIMmrRTLSa7OqmzuzrSJEGc4zlYZlUksYW2iwpt37Sa5flILAQwcnPAbgScKDlqxNmrZ7g3K2blHIZuFG7rxdNTKY9cVDYZWLgwWoi2RUNUGRabFKQNLASrfBbRwznYhzHaLq/wqR8YIv9uZCW8gImSWfU50jAEoe4E1fbq9G0KqnSCLp/I6RXMN9GpeX6TVTWgJ90MsagkWZd8OIQnrgBgZ0zDHprAt2BbLzPD/GDVBbPbbfEJPUbACJtoma6m8/RclgT00nl0OBgeBU8C/AM21DQM2fQxuMTbNQxUFhE8OYPE3JLfb7M78StSHTUDu8pWU1h7b2EFkx3xjLFNgVm53K6KCkhglhHxzu4evcL+RSGpQFmSz1H9WqbTbIk/jDpHluKSPHFiYetpUCBQDCQxzhIBIXgFSmlRFjlY5oAUlRnLMpO4AnMuIRlRcTx6ilsYi/TJRQmDAboUojUK/60ILYI6Y66bjWmiZZGcZr2eh+bCZhfMBRrrFvAoiNKkzjN2E1Su1tkMemN7r2J8K8wMhJMsyxlp2xyggKWlMcHJK5qWAQwLPsdL8kCo10wmk+R9TW06sle6HL6ylh6Et/YrtvcgCSU+NxGX1YSFBChE+I+oUt1X8nM1ZtDRuhGPmh4S7zi7AyqIb7Ln1ijJWoLsMM/LKofRvCcDR9to+RqGBvkaaR78hyO2oG18HyWCzkkPJmRVLj9ncp4Tg8NMRWsnIx5nwq4fPGFPFZu0MB6RqfYsCKdgbyDTVcCQeZWgjOyRz9YpBvx1u0o3n3g5WxbIvutqnCDYN00O6oalRlYRusPyG9e8JPLUW5gxY7GE4r8T0Wx2BzbNLK8TobUCvQSM+CarI68q27ProTLr1Grqt5qL3eaQRBaocyaKqG0+ik5tI1yoRW9J/CIxBsnAZgjwEBTu5Co+3FS+Mc/MXEO33XVZv0J/l5hxViHS++UMpMmeBSiw5/NsrjYzqjj4kNWBh+REpQZR49BqoddCOBsP7Ib1wmZK9xOCgimSbAsw9AvLZeApO88aL2uQnLanwQEpfBTx+t5U9Q/rHv3Qc9D94nEUNBBFmy/x+XLQa6KQbsUW5QghiiKEDUDf9MEWH8gINIogBWgp6e3HVWZNUILC25gjJbZF/ss2Ew0ZF5PSqNVkORfbFlhKr2hatJ67U9jg65t4ve/bWdZh6h4tUcKjAC2jbpDNqbg1Eqar8eZmWU4j1bUeYmVjpEUUIitBuwvqjkmu19gFKCPBV9XYQwB6Uvkl+GrQeYiDi7rGPTXcjzOOhOCficHUjkLsshmgBzAjiV4v+GNwaHDM7tJZzT0nhkDjBmnUN0UHqd4jvsFoBLPZ7CKHEz6W2H5bSz2seuhdz9qqjAcTh8jvhYxTastPYAnlsHSJDdwAEeS0PQdNRkB6sFU8Bt+eCwyHnYumecWWTDOMiukFhDH33k6cZfJTdl9pN+8OjzVzLBkT0Oux9pfs8FM3KjxoVclGj6T1uVeAWMNruty33jN9wKLBmABPenZpn29flux71gVkZPqKnOyry7oGLDsty6Ueo1YH46R33mygrycMkSgGE29VueILOHtLJotrd/asYkCHbIbBBqOgff9H9tqm8UI18cNIQzlv6Bdi+vwrmkgtGogQUn/ZlMUNFzrY26/ULkwnjeVX9euhH9wAr38VODwEaa1MhUYD1gwkisnJd4sbDriSLtP7cltbDh3aYUjsFQeH7PzAs3w4GrNsisw+eqltUSxJom92my/n0lDHX3IscSg3Qk9H0xQ/afgT3o20uDeUjDR8UBoM4qhByP1So9cbNQbtgPaKae6rhAibOoC03TPpABLUxfVGkBcn4u4V3kdxs8aLr+kyxz0UoCBZDhGKwhjVY5zFUWRA9OXWEN81widU/cCC7rXrDcY0alUekAqECVLAQuzcnSIui2nofQGh55kxPlura7lfhHZ8kWiKGHSBpWhuaHDkyZQRCItss6Gpb3iJrSPP9LZeFeLDfVrcVPB/DbxB2im2CgxA4TxM1NsgfKVazFyN0fnvSeoGWg2FLXwSNvVmXaXFhMbNsLzYZo/uWr25b2L+bf3tZC+xGsyydR28Qy9VhdEixGFNQFZndqqhlgZq8arWOBlvPbM4BoPKKJKurDOQv5+k9ql9jVkNy0FiJn7Ed6KVpNEhdUayaIejNQxeqaJr7xIqFk+4w8Px8o+QEu346aB7wl+XPIqQeq/0ed6O2bXADZCECFpFi3I5zzbM/Ywfx0yXVLtAwoCxLCqLe5UWIgCyycHpYBs0ImTQuNpjwDmj5a4I3Agia6CnW1gLE7LcV2mRL9D5KVzmzfFmC4ZxgoqXGfIAqBd9aay1BgFQKem56x8QmbjTWhKq/FKJ7fyxraeje3EsF2xQa0Yyqi9+T38iRKNnawbMYjRLEa7/jldfKgu810ZpsPzalvJgL/sCzCN1BX+lpp5g2aymf+fdFhVwtL2KMcqIVayNvxt6CeJi6SasbTnSqARK2wqZgc0ACnt5sIgO4EDGoDdOaBa0xdAPEmyjEabmCH3T3JiqIXg+F22kHeLaDZvh0MGHMyAM2nurvIjQBObNDcCcCdyXh5OexSQdEwD9ijiCTd+it3n7ZZ/2fzug22g4EP/vKBi0LJ77DAz9ETTcPULKyGyOlFVMBBeRriRqSKcDPZCHwekDOnZx7sdiV0gU7DulbFOMi1QpClSzXt1FbGmpvUy9xVWX1lrTi2EhQn/yXdRrAgEBFlNM9dxvmH31vpVhNGq76xzsN1zqzr9a7BwLX8hXMx7woNB8CPvtoGlyqCAUDEYgu7Wjgr2/d66Js7MNxArKEwWJidor8PiwZj87KootSVpQIvrd21VYoWUeOmpon0p2V+OxhLJQHbKXWD+IhxZzmTY6kbfYohchu/SsTU9de2xvuOptWHtkxKSi/S5gT9odjeabcs026cUubF5U2aaOMMpXLfdJuQEtI6SAjxTW05uMZB6VB1EnomOAxsa4E6Cy1bq+71x9HPtJNYgSkCrbPjyCqvb0ZylGt2FAyjzdzJPlFnd3MdCtxkjLSAbG0diTDmGCVQRGsgBuaQJ3wFf0talqFNJ8OBjIHlFjUPJ4MBjEA1gfVGXoqvr5f+GrkD1DKvQ0GMK/T9R3EaZABzsobmuxLKH/UXt01xPZ6h8AbjzoQaOHCNciiMK6FscKAOfh8bHS24COMNNqEV6tAgUjmnznIkQ/pn/ZdiCF/fRF9M9EkwnGSx7jGMH0q9MaxEIuIv1D/hkYBJcsSwvh31uijED7KTCWnD9RST2k0C6HQ17EgfGA62KN/TgOocCQHArJsGDJ2dTrOcg3oOImO3K9/tBD99vJ0WN7VU5hfn2m9YyF72mo/UawWe/Aco5URsHFZUtzYhGImUmMe2j5xNQ5pdG+s8YsV6sKq4lOAEqrLC2iSHfhqQHeC548CYZMuTAIq2UuWmEshmpCxhWwV1KfMRXtyU84qNqKsCuMAJISIf81S6YAoMJQ/6TC3WM9+7XRiaET+jtM2sPhaWMEw2WdwPvQW2F4fNKsAKUTmHr+GseHw2YNdAwfHx76a2BkULMKwEngy/OQkwWEP748CiUVVtlNus7vYPFCUugn0LmJYA1imBLQ7iA+9BID3octFYbHzRpQPMEv/jqH8cBXZXic+BsZBUe+GofJkUOIOtFUSDGspE5uNuV2HW1wprf1nz5ST86abWCoC6xadeiUHkEnPJTCvZvlfVIBk24yGznQPecw+daZwpALZRoqvWD58URhinGf1SIvQIrpieDBoqyFyKdd9U9F+aXQ+OvFCiSXp7/Hg+TkDOuR4uKrderhyxeD5PTIV/jsebPw6XFy9txb2DMPz46Ss9OEtsR+OTuxKXp2Bnw/0AOOJz+Nf0L98MS7iZXOCXVkT5P9HQx5Qa6gXcVQTAM48nL9Ml9p5DBUDStFoKrNQMWqcL2YZ9VsFF5gd/D8IOpHQU0qEZmt3P3J99kB/pgr7xMjiBtezP8VvMuXaNGjew0MuSoQm2DoC1qtsnkOsnd5H6QLWCwDjHf8R5iPdUrH3dIZHlpF4e6ATOefcR15Ns8w8EOcK603Wbqi7oujtFVQbWe3GDf57vrH2AJAAZZCKaEISzI7sCShlotwOGG6Nw2yZplY9Wi/RlrgChcnrJcCfVw/leoTy1fQt1BNsl68hYVu4zH2vuRzdIbeZvnNbU1BdAgCF6xGUdx3sxrCF02IWk/VB0Booe7Q+ZowvO5tUlE2eVbUSuUTaIDOld3li6hH2tfwxVETnnRhX9EfS99rB04hbG4RdOCmy/VtCgXCi5B2AhQSUxSkwKlIeDpISZEKs3tWKi8Wpe1CwLVVahfWUNh9WJbFjViBR+ie6Soq1gGlGFFBUNCxkigNOljPrsFWNLvaE8UWoOAnYF8kA6Pma/8EBitO74Xyh+aKMDKBijAeVU2HbxwEu9wLftfCkydQyW+qhut8KScBWLjOtGipMiuXsJAg90IV/NNSTg82FKOdef2ixW4OkQ8TxkZQkT21VBL5ABKiufQI0O9ed3kxNrKC5ITuGvIg1CL8SvAf7r6Kam1ul9BofrIZetHWimZS5dZQz20VpCpEHIv+GfzbXZS0JijaVKLa2jDMjWOtH9o6LBV19DBZOvtO8LJWm47bVt9MHxwY/dBV+lM+DaUqFrHZ94xUc5ieLXWpFPlbCSM/CC1b9NwncdEC0lIUlXwHwGoF2KMWrA2mAj7tU0mQuk1N9WD70BJ/6i4LuODCO2cXWahS31FkZehgStCWhm7QiSJotBcn5NRLkq5a0vWJ5ffrJmmfwkXd7bRzXWqd/jhJk8TrEJQftb9CorCPz+1iuWzolcEizTEhDNtetaxvCb5vIyWVb5IQyrPuP0wkz21IVRmoO+kHeocQl3mqG7bt9jH/X0ytTe8jCRJUZiBbkcrjQKRVRTZdsaGR2MjhW5rskKy9sSk76/SBew3xWC7bzWSncNlbZ8uz9SCuWPfkVqfPPHEsC0WTdJlYXiGJtjm9E6k30ik8wd0Splr8H1aCrfBULMQsMCEvq4v6JdIknmb1lyzTIgJp1OekQeaZLbd4+HkUTkt9DkBaSZgJ4hv6c9hr78cPdj8mrRu/urpxpU9iEeSoQoHTzzBD0mkujPRui8/acBPTouIBpcATbLezPbSLbZ/igWoccqkARs7wW5gjWcx+X1xtVxHTRhESEltBcgj/CEjbSvqzUbFWCPZ1A2wXm9HOK+Kb4t2/6dMUx6E4Bn92wroANTQ2Bx4NqqqWYMQmTsdlLXzrqSX6ajZVobR4ZZd98ISn+rhhrF8gIwhIVnyjpw7aPEhp3zfjx3b2cD0LwIHvFAqJRJ4aDOfN2YnMbRUHfwEVswg6shQ8gxneCFCXpxH4dvUO7M2edFNQiu0VbTK2xXcEfwjOTgZSzql9XDllLU+P7aLZOQVVL3DWWB2js4z82ezfC5R3cIAd38hkIJ4ecSZ63xWUPXdXnUxdNp/3mdpOgDFUmJEIptPbCDKmldS2MWe3ZZWRh0CWlzvz8pH25v1b8rJEP/DsxasBU3JCKTbLcjaW0lo03JsoHtM8SzoSPM9A5VZwQNreFOUmE7qAUK8et5Fqttv32EQ12IxDj4Ih9b6J3P0xm6mmrLWjKjZJlPfNOjBgZwhBpmRvmOIa2jlCrILpHS/YlSqETPPWAhyKOoHUmuKFzxV+oAGEryB1pWxbmy49pzAloRIlI052S9mRa5ZTlZJVdVc99FTl6xgayOyRlWqcvUafScfJbD707KSCPtClvvY1I0g1ViQvTFTMCfqDYL7ms2qnGqs9181Ppr3ur4lCxj0pIwqy5EMqNtA5QoNlz61AQgzEGvGaLBBarCyY8c5fjPIihi64ffMTKLj7lDc7+c4JHiPhRfSIXNtm1WdUYSwbSZtTPpkjYlD4mZhQaLSAIVAun25p53oHXAaqbznwLMAG532bsI+67O6H07hREBgW4jQYOYO+uX3Lwdd3HXh92z/nkNqLlPJN/Qaqa0+XRXPjvvrmztrN2P4xf2+4G/B7tWv5C/3NWrbid2qWsVPf47TyI2Lc0d8LDebg7nM3dqN9aeWhC5AYApaVlUiFYlpSWo5ybnTNX8fjoWHE6c1N5ARvQndGTp4OmgCOYwvjOtDnCWW1+5MYi4JYnKLzvKUwfmgUz71l82bB9M5XEHQWH66OoxVqua5XD/qMVFyD48HTQq6PQzNaDW5B3a0xmNqKUCdR+zLYCzd9CWbzrKA/F4+M5eJr4jMNVa4+mMA3XS7t5Qf1GuGbk1vFRomS3NVQg5Qzr1FBeRe7VC6/c2m3vuV34vjrWSpobkSt7agptKfGqS6PqZrlxQWza9mYTGKca8l8iyfR0IbCDbfbdJ1RlLfTSbHJ7WLKp247qjJoxKWRhR6Ryv2kMG2hYIFZGfeC/cOjYOO85ptpLVBjNPntijDP96qY3jkVheBRdcXGSmttKtxE2drPa+uuH+k9q7agrWv78Tb1PYhT4onHWhVqzltroJtH99yn0PNVy+RcswVSKEUQMzpiLAZTSH6RogrtFNKuoeDsExjvJN/c7QESxDJlkt4qoCdhpMuHOq+XrZmVSI/PP9Mx/FpkW8UnGdkorH/6Vzcx0lsIopWR+NMPYCHFVNejENg+pAwYS+EtGDHveQpmAXocsJEYuxh9AvAjsKJRBZbdlSbEIr9B8TOKMB/jcc8GEiNZqWsR/ev5ek+yMgqvtzjMQbnQ4UB2WWhnu8niGplJHX/zF8GBhZ+RSBQyX+ejwxPWufWyjmfLssoiVkkdX+XDacKM1MrZYkewk/uWBnfgC8C3mHZrpzO4FocvcEmUcUXB9D6Q3hrWgVLzqkRpei8MM5EJmy363R1qtwMmHmvh2ztEW+BPKf4E4FS7OqUarL61X90WwaRpOwg9//H9u6DqT0XMDAHZ2TkLt9b+keqvYupVt7otjkmzonkYs58tG2IT7qQ3xd1tVz/l7aZtrbZpxzjfvcT2EpxeXsl9ubm6CQPR1/GWHvK7Q2Dj0xwCpjODgkTKVadFY9nfepuWO2LTapYVlNZSbuViju5oOLBLcW1dimCNgCOGb0NH8r7oyTVkFH4s18FwoIgjDjwLDdGAJtF71y16d4hdr8hlVEbUGWEqmaXfksedsniD2kj4b4VMESTIHkir5FwiKUqxNKRyjf6mNKQ9u+EflSL+jAgVyN1Oq2U1Nzs8ZBOMhsXJWtzw3I1ua0r4C7nPp7e/xdbVaL/mkEski9DwOI1ovutkY9NSz4qpkIPAA8L1/Qieczr9oCMzESsnpNB0u1hk6CfNy/hPaBq/fitZUgRJAqzPeM4rfP/jn3A6Am+auSsq9/m2FFBxJESgeS2lw0hsiIl8OlbWdG4isYzpzQo8nTqvpHKpj6ycPXaadV6e5Vf3VOHZ11WtHqMWzPfskzpZ5AYLiyIULixinOeNeHX53qXtrFzf63Sh1i0YbaexvstoykHUg/fu+kfUzhVFmfPov673mOgzwWT8wuWAEVKN7YOAvrMcqo9hexfvZlw8dkHceMSSRpD4H8n3LT1Qp/sVdiIF7s4bR5r5ubGxJjRUMewrUho1PbzT8wPi16o0wPgkjaIybcDj9CTCSQ+NyOMmdnE7sGcXtzyuTT84j7X6n9STRqasRfiziOknlM4xh4/EzOT1oFc6bszlWTTZmzFw/p21lqC3Jq957sF5FOeZMpynVXAdYOyIKn0BWYK5NCN1Lsa/4UcJw9ysHw5Z5NpZTnUclThxKsqaFGzCheHCabzV93ZoOK1bjLKTX7lLx9qRPze84u7VW5tC7Og9xohSBRYgYHl+kCIy+N09S043kqD7RRHINas4Qop5/aYXQ6Rt+5DvvuvCzIPKy1qn83Vh9tYq7R7/1xWc0+e8jvfYvK7o+WrVto4i6Fpt+5uesH8qb7sLPTUsbx+von2ibbtcLEab6rWES3L/HkprNi20l5C9s0ggJiZMdc4qzSrBH9VREz1TLL9iY8YhQ7rvHJ41kSymipmcynV4cEChHyBTN3la1B9hIMuNNFEi+VfKJ5Q0SYKHGpPEaKBVtlww/dMTUswOGjdljEk56RdWMgubvIPxXF/NNx4zjQJUEXHdm0BfVuUbOIBkLM/uqQz1nkN9dnm2yBnUnSKEuc6ibX80aKPzUT8cMGIuswJoiaVJHsJYNpYC3OTh6Pd4/Zusxs0qCUN7Z9VZap98lUsD6lAM6phqThp6lkcN7DvU6XNSNJL+fhW5VVEC2DSRWhGui+UXsb8np4cIYDOsiXe3Sfb8b8aYlJlVJLsXCRrtLKxJV/3fmztl+nfWBfs774G4CEA/2gVF+F65sYHFOGovLz++w2y+H2SRyDnkWcxh6MnpQ6mQWBPjcD6rE1YidDPPoqE+b6soPjbq3GxSugarmFctFVkJq3bvf8ispRwXdsILqavbyS6eD5W1FTzDVCbxwHitSQaTsw9FMiXmp8tOI3EBUbzONqttnUVDuqHyEAEVdX6zLbcV2wJZizTQ/UDfWac6rxguEi2xeYF3Eomo3wYLK8VI8pLbDFR7RVc1bdblEu9ycRNgES42A5KzEv/t0283+AEaH4W4vuOpeMd9CyrGDVoCG1DsqhG7Xc5kk6tzSnI4Mpen2YEReLFxJ9KCOvG2qH7ZZtmvWeSehfudO2AoH6mfT41oiinT4fXVx+Snq4tr5DLPpw8fXzr0iPDPbwEjghWteA1zvyOeTyOmxrDlseZQaBNDffLVyCVxy9pmCj04DaurhDTTNxZUUU7lEEF9VNzJorZ9z1UOekykpu77E4ebAnbXnxE35kJcIWpKOvtXtV51ye+ylDvb2BoPTmbXDGKnzQvLQLvdLhZL53ZFz5WElPTJvLEtjCIRt32bkdnOU8y1KCN+MZ8mq0AhIqBU1HarOuG421bwR35CQ1JGjdHXECRZlVcw0WqGr/TNh5SpFzu+SGfCoBs+WHa8IXz05ImErX0WJRqqaDgw277cJBjhD/MqkoFqaH2xjNwH6lxB7nmvsoKD9Yu79Jk83CcxwgsddxxvU5k+fUk48oWEQXkwDWK+iyPoqzgBgJds0SVQpoaV9BB1Pfpkme6wNNExUcxTS34iq2u+fLEE45vOd/Zl3RxPNQzOBxPHych6zQnv67f87vSc1+r5yopfY/W3jRAx3voXcUr0Ju3gGsFSVuThpyxbj0KaJw35I1lAgDlofuBHX/XNW8Ui2yRiQSK9JDpgVxUURfwT6Yxd7qkOl9M3moetFoHPGujS5OfZ53yWKQEkntRBUlsAd8bbN/2KbOr7I+n1d0o+bgjnSAdhfnJQzzDrgXGV0kZbQ3jsrqaYSVfn7Efqmjxk3D59DaoyfD8vhMWyh/yze9m3kO9buPTtqyTk/bUTJxmvEEzenK5iG3ERvpeCUuMn718QBHlmu599QkhwDJ43E+RRfe4QcDKGQZUUEQzCM44Q1JqySsECJuIJkv+/bsoLeYGY9CZdx6SMXaqbaLNNlXsJyu6837DHD4BABa//SbfItprUrkGfzXuZhVDcqTfyaUVcGeoHfFeTKxsq82+RfUn2SStFQx9nsKSwW+lEQzJ+jR5wTyCDdtCJy5R0SjEl0JTZpGB2MW6C6dbkJ55AHVVSHBhqRF5iQ/vlkRRCAcaUTmHlwmvE2Zjq/Tchskg8Y2pAipGPekpcptu6BAScMHsBm/I1jMTvGH/biq+wFqUFiAbj4YldIKNr1uccgthLAv2Nx62cO4l/bnLqMpFJ3JdR9fimsnSp5uICDGq/Khc1BnuKygId7CQq7E8Pe+NzMEJhWV9vo14sDFX7mKfW4ft8RwQH9dd8HUniG0V/ou/S7dsIuansyZw3ljw5rwWMnn3KU7Fja84N7/6PJ4+xP5WGb9fDOpvQ3PmwUh51OMDtQ6it/upGa1CgqzE1d8118BNPo72WNCd4BySbpMEfMDPB/7aEguf6c7Ej2ci5vZrijePWedaxktSONqSGsTfxnXS1TXPvgQPVWrcia6+D+nT3d8WTwdNIPU6jlMq9qt2x/P3Qpt97c+S/LtTSzGZFxW5Kal2pfX365oww3QPYD1x7hy6BYOYDDU3lsR7E522xzItPkXufIAG3dG5xd9gOjbtLuf5uWrLfV77bT96uXe+vSJuBxwQ2bZdK/l3d/ru6/T9F3fbsqXUr22z69q1Z26F76xHRSjhfYwki8xXiXRztijljnW/WzrVTVV3qGNk+R5ZhhgdwgFZ2mFRAeSt4/ja/ufW9x+LDluL2+7+bDP91JgOsuBld3eMaD30pLhL13enz9zcpKphnoLLMEtCsVig+ZMMgOuQXGBAJMcYyEULFbbI2CwWDbzzg1JfHglPrnuTHyG//9FpYFP97pF3UNnT7W0hm1bJXUb815LWEuqygDgtoH+tnD8un3erZ1+LptRDEsJVi8WQ5FEejCbLFjmM+9JN2kJrpvCAtluwCicwGopgu87FksydzNCrhWnBBmYkNuMMmptJyX5jA0BqP7aL0cAS/WOhFP0ThTlPctPJ3+/Lv9uX/JPsSybCtUeGsN5jIyJtM1I37bY1vuU/qDc1vqUXLQFpUfcUK5STauqcoTFO+EZjZqCra15XFla4SzB9HBtOe4g7dVl0APdfwP/yt1zKcaovxrngX+ia/iwT2fQlY3q5WjcYY3jKBcYTBUjqoEFrdm+v+/FpCj+/K86CIuF92B1Pal9OhVqnKaitD2WKtYljX1mtVemGVrlXp2iqdzmbbTToz0bPqhVCPHXL2PInP5kkDRuPLbmCUsA/Ui3Rp8j+LR29dEGNlJQJ5RzCyv2YbtMs+58gJo4GdBwDZ8dsgH+4BGerMchmOrRQQ+ea7wF8cmnzYh78VImWBIVK3Rk+LaThm/DsRFxg1AQn89gZ06ANklKjzRtS1nYTBmuZQejxWU0JNAljDxorxFatPJjysRMx41IRnSbqdMT1JvvEQuJwKRwTO+2Kt8tuKEj2MSjnkegq/s0o2luKJxRvOKRPL3pXzzi3VgYotL/Bu9V4LDvbRSR6uhGsG5uNBh9Cylsnv2DIqxaIIPSHhKPJPWwEnu12Nj8lop0dej4pP1VbJHWGl8/sG5Sr4iNx3Btp/Wuo7lfljH2eJOfC7T2kzNdbpPfojfAvaw4G+q8Ma413pXSNmihnTCyNxNvKAFQPXTNYkh0J6NRsqClNKjE7MKbVv2uOuQ0N7HB4ymfG/gilyLg0caRHJPCfYV4m86qcyXX4YeQTUw+4Ux42RG9seJEtxxYGR7Y+brU0sp4HJ/Cwy2+ZuFngr7yAqvoJVbYPHsGH7MCJ47yg2+fiRtxTsNaZ7jqv3OJfJ0N1vuSah/TJZwyWtRZrc06Cn4qNWGF385b8ddf8LJlxZoIaHuahNwjTLZuQT1M4yMZaD1peDIoX19J5d0+wAs3mkNcewDbfPR3PCFjq/KabzNrBVgyVjAvSkEx1g04ZOFzDeGRei6UwnTMoMZdd0hneW6cRQDSEhU2QA4tzo5ca+O7Stpr2/hw7o/o69t9Apn2DzsstyYy3fYOpzs5a1LDhArzqVk2O/XQVRVCaksjcRtFHSabG0WCAt5oNHN+cApFJppT52VT93u4Fth5prtHEVGg8m3CfHCPzA43cp9Tk778whHLZBkIIFmxiNGAoPzgrwmw7us1VO3IXOX6IzmOaGNWiGlG3wDzwictcB+OaVa6gqU1c6vLaEITkmbSVIY+6kdW8pbnQml7K6AdzhdMZHjar/8yNobLngq7FqczJugBjb+U51HncrQ3HD2Os3z9Xa7l/cYMXuRx6cRTq5luXfY1eGhH4Th5bb1xrnfUV99/3+S6bYP8Dk83OHrrO96fo7U3D2Gyk4++0UFMsBbR/A0rjKNjdZJKkKS00xsuh1W34ZhXlRoA/WA8W6d12qCGb2ddU47Et2kApFy7S2Kw2NAtIyr0X5pg80obXNMfzdSv7BmQRPdQn/BJjspPaYLdqJJPY+yDQ4E3VRC5lu3vXhYu/Z0CupKFiit8Pm8Am0ViNktj+w2U5gDQ3lP82oMbx5bnjzNxlAXfbMIhQ9+ypEyoN2xnHqjcVH5Mep53Wv0xQShYwRrZS2fc0Y43VszirteN0x+ZruxyZ00gLbm2jtoxQUeDJJocEzPHskv4OVF3Inqi0AfyOqh98ZVSZ4Fst8jdkps8aI+aRT+2j5VhcpW7S5Kk0hJnJ8VpDYi9NF5L1D9rEpd95XrgnbEAo9604qjZib7tJvfZnyrnUpv/BxqW7zRc3MS9mmhe5+zVpV3JbFRzo9vsIt8hxTaqVfuJFHtgxV6vNs7pg2kiwHcx0ds8yMlYYGM5nKIpNjeCHfY85NkpV0qg9Tpc1IzvEE9ZFto1kmmwv3FV7hJj7tB1obddq+c0G+f3v59OLny13gPPmWpQNCZFyWCRW1fiF1O63WyGTLgsb75lM2eTxBdTnp9YP7Zb4aRYMYd0PxQlaZ2HNHBmVfo9+SNdmk8DQs8qgkyvvwt6KuXegxNFb8zriqi+CWNGLUdy6jtkfCTg8gsqtekugjby1jU2CSalsFauLR4XppDfuO8hPR0rtbTBUgBpqu3ACclunsE3QSP1AGp9FhR87sl0gCjkf4PYffmqJEbV8K113cYK4EYRv2mFp9W5GtIoNDQje7GH6VfhI5BZWDkV1f6WRw8tURPk5vFVkGL9mWk7wZOMzPwvsFLF0bzrm4DchDp3NR0kl5FOWjk7T1rXT2yqbsnK3u5oxt5o7bNmTaNmE873mcheezx2G3l1dvh3dvh5evw9vX4fUjo/3AmO/+nLl2/P1uqaZTFtPEbBcK52HPqdRgodYMvnKfWN3OIjaKMb72S7ohhfAzCG0gddS6hbsGGYN5nruDatb39S0FTKyXaY0J6GLxJsFugRJo5YlQZazi8odVkMKBdfKJRAHjt1aHFIWEd5auWwqsYfVOcbKDitdSIl8uS5ya716/aRYRM7He3PM8qWsYF4GWiMo4YJFegl7jkH0my5w981bkWRP7cnAeOVZVrRhgbtdkBkq4r339kVq/yUCM1ptIvwVFh2GBes9WJPuU7LYHSsDmrXlBzj0YUSy5JkUsG6fantI36y0rTE3c4MUbItydHGgDi8tVVcnmYO+XS5CfdBRCB+KacOGoI6Eu3sb9H+zYTHaHkZF5rdN9ISpR2AbaKO+qIicHnUbCyIlIfbXCW4WN4c9yYSInX8GX67J+hVlfRfikneNCzfyaf7DulZaLn4WYtQ1BmILuQqjqRE+F2jnC+3A2N8tyGhlWS2Da43CJvbIY8/dbfbP7ZUtMuubS3OdKpuu5rz/sJtfBpLX6H63au67//Wm7rHNYq4KOrpCOi9GoQPJ/DCjLfMv4Ny4AZlk8FHduiwRPz2zplvloW+E9oV6GlIwII4Z/WgW1TiZH+sTLq1cXP7/5mFy+vX71+kdNJN6Mk4hPJd5hRXQ168BZtSo/ZZRBmswiMxvwdqC26057eBTNPoam4HVVw/l/OBhIGoJSVqFjUuh86l5e7C8lQsPYdo2l1AtV0gEp20xVcSjLhtij5FZ1dPnnq8t/fvf29fXH5NXrN1fXFz9dfTAktCC5LObNniyLBl/llLNB9FQMs0GH9NPu/gkVVnWPTtJDdXLEP3ZfjxLtyLpyb8pBxU37Y31kLnsFxqa56o1NcnqrKH7x5k1yefHBQ2an/R05qj1EFu2YOHGbPegIjM0FVjlJZqsYvZMGS0YbH9bpA33WAj+qiAl5PTe+osu55WG7dez/INY8kCtbUHKdb/utuqwIA4NeKAnKDeZTa5LqB/ukOiG4bCnvCkEuU5nK+iZIr68C8fp2YAE9Kn2Ts19kRyOoBnr7Ru4B82EyvF/pYnotBY1ySqE9uG6Y4yl4XBTMSlR86WVfXKAlvIVtsWJS1PEQH9dKc+IqZA1lpTXxdENInHx1GpAwgSUgXegrM41hOcHTpXSxZZ3iBEjSGi/+w8BaXPRi/CfqPahwBY9OWbSxgrXmH3gOXuGyLk7VCpXL/RS5mTB5Ac0UjbyA8kC9JzyQpdkm7cZjxum8zOKHUDlaC9MtUnijhIypJ0UFyYuX89SJJ4PjQ+OwFaHb12qTS4RGcNiD7cH1cRCbMcRQGqSJJvJRsvdoyNomlEq5hu81Fn3wLXLs6Ab6UpMaRF9Fxp/D1y0hoZRAPCuwMkb8he/IbHu2zKf4xWfJq/tXzoO9rmYx3MCuYfHUbbulxfiK1J0s58GeF7awzRVzN4uvtu/uFtuvshZnuESOAUkq63on0B+FM1vfGIGvVMoG6fI26Z/ywktb6xKOBNQGxDd8U1bVEnNy0+WzpoV/RIBPdU4EuoQLj0Rh9sZKHsaPvW4ifZmGbuOaXeSFTiB5EVWwSu+DaRbMyy0I2KfoK0RCZXMX7kMj8TA7Ra/k3yIv5gngek8vbOY24rBv6b8cBJ7kI1tFDh7dIytGpYIBsgQpm034SaXSmGc1Xv5m3kW8gZ3grWS2Ol09Rmls8yX0Dc/C6Xs77D1x3ox9slHj4rwHjaLvnK3c0y7werzVXdPy/t1AXslq8OVd6u3YCNPQHKGkdBf8LdIXU0m2+fbbYNJ1pRaZ9wIt+9yNrnVhsQPW6Pp937WmmgP4Ny8fqE46zG+kIYNA3lh1u8PKuep973rpXTMTeYO1LKHIQEmvcZLONiCF+J3JEy+X+cnvcrEmZcdwNCIU7RFva3vnstndhr0J0IK3c5utg4W+tBb9WjAUtSAITn7zreUeeM2nO/jNopzvzATFwtANeCYOWKlUKk610x4VJ1dGzlVEYjNSo2sw80QTyQv4Hhn6I6N0OmJ+9jxr2qjDz53aZ3Zbzp82IPCzqE0Ih3tAaNw5LsHsunHcCWt53G3NgjT7hx+y0UsUG1ihHKzArpnfgNU95ymnhAhsZPc39pr7O5cYtStKOrdCmmKNtlt3dZiYX63oDAHmhyJHpPQsKyNsI7ydbRYYVUffBW0lCPNI6JjTPK0SCV4Cob113V7PuqPDhdHMAMagaEQNDOHQ145+8RiJzYJOZ4cM+56tt7aTmaew+UGlsDl30m3Q4Pzl4v316+sfz4Mf3/2MDkqgeFncLO8pLcpqBcIARUeGt/CC6Q7vUSa9gh5RiqfYGsbHnbMUR/g41iE5ncV9ABQSBLaSJflch2P3PhCNMUKUVmnSF4/WvTm6Xiylv4hiTFH3VdZZ5Ynm/9f849M/PX0+9GjsAB+qYiqDEVdQVdvym9d2EeM24lnoWkK2ERjeiot3aRTxG6JZdIyBGENPSXnMRnMYPkRdeWRoxow9AzLxZLZJoUwpDK4R8WKzzBe6K6tKymJ577vOoaV3hCgtxJkIEhBTMM7ugIqz2vrC+tkPQqwtLRD46klRwilo2tOJqno8sVeTm9RX5zNeJ3T55vW7v+T17Z/xzmKdCdFziMS6Vkjc1DJNZ5+muEXAUigGiBh74YEkAkfArIh6sYboD2mk6ylUMxSwLn62F5Y0wj8H3j7AFPmSbub6shlMe3VuZUQXCf/ZC38XWAotC81YuDzkZTNuUt4WCPrnM/3TpKd6eihSuuBvYo0Yxm61RrU9OsyeHg794NUOnCJLpCD3PFKH5WOxWIKLJM14vR2ct7cm+Ju1QUeUjyOPEOgTLj2R9Miber8hWjhF2kK+m+3sSvRrvGpeWTVWLtBJvx0rsxy0pATetGq3HTmNRVpD7zefdPfaiIidncDQW4epaLuSFKNfyx6KwMuOfWvZsA9Az+IZss2sjhrHv/bYlPFszlBUEPAOjCuXqKgQFOsNJnMosvrwdA9dAIsL591IKmvx9bv377PqGgFEzqzG0t+6MFqIgWbjWQM9a17HkqdRf8SSxzrQtdJxqphmuuUNgeZ6klGG4ssSVC7fRB972ZPVhJEARsbMR9FweBQ8CQ7jw+M2y4m3mOElVJebco31dpf/WIplJtpd9BqvCl0iVq1nC8aD+Oj0uB/An+MT+jM4wbQsg3g4PMNnwEn8OZ7se4yg60jR7ynrLb5+vJjX7NVmVVvgHyPd7Qm3S7Db7PvfTK4jct9frBvSc8H2+4luTxbs3QJcpBPXHPBosexp9L/EYIEOO6jL6DhAPsc9DFASQ0oFePXXd1eXH69eJhfvL//8+iP8/vn9VZPGjSAPPwuHPxcqtyDZ4wFv8Dz46kPqH3iZf+g9hDuYy7mok1Yeut/OF2Dlq6TCqrwUEkVA1Hx96O0Gsa9f0heQeL5vOGSbJ4/MjwJ3hUgOkZ3VvF3OidCmkhYWdI9V2wFJ52a7jlzW7W5L+2o7f7rrxx1vVsm5UbhoullRC8QQYlNFZD/mo9eXorEJVCg75BGhmBBRi1Sc5lhzLBxektFMDJJRkzqP7xqYPi2oZY9i9/a+zWHajvXsUwgzu0HS/n9T/cAjYBtagnMnRMcwPkYB8In2XWrAd1/iEYmuS26+VT145FrfMRN+nwXfnFpyU/BZSXeUOGDdtdzx8g6GxuWoneefhO9AXzDhjflySvPAPhFCxTFyClsxDaI0f+UWNwECWLglWiA0e4sEUj04pfTJLldrcnsvD3M50sCFJsIT1a5ps+c80WDjjJX3UtnW2CorXM8MbfP69idP9Me+bhFRE78evAcxgrTCd+eP3Ez2xxC2xFoZnlvA1Mjm/iNU+GUflqPiGWpydEsBFsUcwNAJ9Ngig/DzOHYNOVBY1lMEvQIZrjAIU/2O0bgFtKCOi8mD/7JnUDcPDg5y9EgLbCjxUJJgWFKSyL0jMalxg80O3efH6pC68Xy7WleRKN4PwGrEHO5pNctzqUvTneRFPRqibrBIodgIKybyAVTB/w90JwWBzvQAAA=="""
COMMON_RUNTIME_GZIP_BASE64 = """H4sIAAAAAAACCu09a5PbxpHf+SsQVN0FkLnQLmXpfEyYiqxXVJFllyynrophQVhyyIWXBCgA1O56j//9unveD3C5svK4VFyJFhzM9PT09HT39DR64jj+c7FarVnU7Kqu3LBoWTdRdwF/y2u2iHZVy1h1siw25fomelZvNi/pfcPYyaZesHXEPhXrXdGVdZUNBu+hIRTvAF7ZRmXVsQrfFGtoXFYLtmXwT9VF9TL6rl4U6yyKnkZV3bHzur6Mys22brp28OEDIJNrwB8+RMum3kC3AHRZrtkw2tYAHHrooqKLPnx4eEmDeFhW21334cMwKqpFdNWUHWsH0Hm0YhVrig4GhM3bqKuNRld1c1lWqw8fskEcx4MB9ZXny123a1ieC7QAJCBK+LSDgShbzeXTRdFerMtz+bOs5dPPbV3J503RXcjnBlCsN7yzLZRDY9nTD1iNj/hmC5jJ8qfVjeq52m22N1HRRtVWFm0BIhTA/7YLWdbVzfzC+pFVFTWr3NJsuavmfLawwkuOwQ+v38juX2+KFZCe/rwEMvIK7eWaFU2VbVjXlPNWVk4GEfxXzOe7ppjf5O28btiQl32CqVixfNuwedlCf+bL82JdVHO2yEMt53W13FELIGRTXvPS5ZlZJwgVyoALrJJ6nhe7uSxKBbmRFJ+ouRxHB/PUwprYtF6VjBaAGnHD2op1Z98MxdPjU9Hi42KTFTtgOQkSCgaDgaJj9ub7p8/z9+9+evvs6fsXz/PX3z199eLHaBK9b3YM1tTTd69evM9fvXj74t3T99+/wzecuvHLsmHL9U1OoEbxMFD6SJW++el/Thbsk/W7nV9UbL2WZc+BShoO/VLtXy9YvWqKzV/OvBLVgnqssEk6GCiE85dPv3v95jUN6TaI+FiV9IwhUEEOZ8yfg8Ny3onhQenzp2/enLxwBuqVG0Me61+B4YfeClKM5SO82KvJfPfi6Zv8x+9/eveMqJLEb56+/v5tPIzid09f//gCH0pqxTp8Xi4vPuLfeT2vYyDtYMGWUQsLKUfpkqD4GJPUAMFY3KzrYjFGYZFGJ3+I3tYVG/OlARWybdGAAM42l4uySfiPdoKMNozYddl2eX1JP1NqclV2F7xdDaI7ia8AC1bN6wVIpUm865Yn38QpSosLkD1r0Q/+h3hlC5BRiUBoKKpg+xblatHOy3Lysli3UFaSWpiM1NguitHjJzkKa2N0NJ62a3g3i3LF2g6oJ0RvxhslYcyb8zCiqO7mF7sKtA+s+Y41ybrYnC+KsaiZNaxYJGeno6+jBxH+SYfReRynGoLGJdttF6BkEoKXCskDSqSS7y/YNX9K5EDZNQiYeZe3oFlYvijnXWLO4BDl2ZI1DUjES3YzxtFH/0tzCgPHP0QTbDeFV0MhzN8DjetmxnEslzYQ0o0gwCroE0StniCEktJbuz4QRuKkBq3xBTzE26nVbEZ12Rq6D3UG+uY7MhTSw0AzgzLpYZAa/5gks0HU+PhB+G1nR3f7eR1+VleEpdlLz7SK8cQz0fqo0R+Fx2FIAkBrLLWmKFsWvb/ZshdNUzfJMv4JRMEWdSIYZvMLNr8ks07LsFuwf1S36T5OJUMX1U0CPIbMAQYjrvUk5nZnBssc17RgXI1aH763CKdhm/oTQ/4trw1I4whtUCYBDsVPC24GQmPTJuneXO/6tVjnixIsDTB8cliyCVmpeVPXnZTaKOiqYsNogQ9JbuUXQAq+4CcRCBxc51iZD2MOs12isGnhbUsUJEFJqNIDijPVT9as1vV5IrtJSSigcCxbLmRTRVrdt6IX/oIZmkRTrwsDD7Nttq6vQJSmnFQNoZbKwpkCDE0AoYTDT6PJJDqz5aogJ68wPbVa8kK7vkUWXmFgdKRfp9FvrM4a3Jg0NMj4r1Wc/QyMCAwaRSfRLSK/j3uGnUZQDPWSCoVx7DA7mndv6+5lvasWnOktdJfxi+stmyNx2TWoAdgloWS/lfP0m2YfnTMgW3Sr53L/O0AF4EW3zpD2Y8DcgX8rB7b/a/Uj62hnR4Ng19t1OS872phFz75/+/L1q0y3trSX7gOngDM02Lj1GgwQjheCbBO0z8vV2FBHoMJmjoZCHhYqTCooNTSgPr4WgKaxfhPPQGOSpiWlyXHP6wa4t1okUjV6C+ngGiIJJUDBe95ptmIdwktVFSKXQExWpwWk2qKY61vhGiWOTWqyMGwonWXosH8PE8G8Aop7ak+8MJZcmg6cpYPFnHKboiqXYHvkXIhZA47VS6wvoFhlY3Pi5JqyQWrcrZaSdnblNKAgZA1DpnkiDGzgzQaWYs49E7lsk83bT3Ga9qKgYAMLS3Ggyrjo4fNI4+Mydl1u8/NifnkORUGq2TUM0vkvFGv7QImZQp1pfAS9A2DLlriALEK0APCHXy3EYP3MFejG5DT/NfKdKTBuVSccltbtsB/yVq/F8l6DoShalxUrmvwCzPFsGyw2pNdQI1Btm3v179Qf8hK5uc9xS5+386bowM7eeu/DOBTlgt2BhFdnKGw80Qm+jk2Y9ood2ww/dCbA5tFxgEN4i72Q7vM1K6qEr2q1k1Q7L+BEYQi1nPFABSaGnUgvQfCB1SbM1WoLHFhJkKnJhZxj4tg2nxpRU+Aj/HdILKR/w+Z1s0j4n6C+UbgaEHl1Ie/IyEexHJMGN98VsIUCo26O7j/+Nob9XFVfVWrvTZvzY3Gh/X9G/wr6UWtQqAVIBbNn/SIe+i9ibaQZxNZthmZPPol1xQxkGGgqgPvu1bfxXUCT8xvQ/LDrhT9F0xQ3genjHdNOu6yzb7HB6+8NKGl6vz4dg71cmtgTRQgpmB1D/gVNx7swmwpIszCKPb1zUX+vzo0eqfXsniRZl213LOFpSBb5g50d3pBJZz+BQU80kxsyA/BeLYnzXble5FI7zwsQZT0G4TCqd529A6J10u22azbdLrLnAPllQyYTvhRmIjlTsdOWdcrpihvDXBQKrY09S2vS6Ch6GMWiYmysQqqOal03eyicb/xd7EA91nVmgDimCTduOX6AjjkuraCkXSypLCpwOTbTQr8F5dJN3NpUalXrGlZs0JVHKFF5mrUXu+USTAVdjbHFBFSSMstbtmZ0UpDjK+BkDfIc2oIN3Ja/MLsJB5ob71U7Ti8OlC1AoBZr3Je39a6ZA89NZ7T14j9xsxLwoe5tEMviEuf0VqkNBUWVGIC0Z12Agf1ORVvB04F0EYoyKZWxNat2GzpUSsQ0DCNyR0zOjGW6LmD/BpCQFqZAp3LTXkWTlNeFbm1hIkY+8ZQYUirnbz09ZQIA2Jp6Np1ROaMpbJVOeW3QXr8nxC1gJh8SAlucT47EzKrpbGa8SZbdZMUWzwXF0DTq9uYA/9OzNzloEnjDt6bdZhOPAlg6VQ3uJgLWJyKoNvegg9NZHynQ5bVe2zgg0rwe7F/+wFmsd2JsiiyVxdN6LJGR4dUmugU34e7Vf4gmx6GALQMo2CQ8B0QvTeLIJeugsSmuc3wluAO1fAgQ3+2WLSjelRI/A3v5je+gb3RiEcQ6X+CVhv0kFy5E1y+mqPv7YyZ3bw1DisCBt3bGd8/UocGoesP++bvfePqZZa8cDebkABLmKN0d7TseyBB0t0mb5uOu7gpuvMG2eA2cUyw71oDS4Yy0j5r6qh1HrjuNSC86n9yaWO1hv4Pj8F5i6d50rInzZwSPNt3UNo5m6GadyVNrJNdd1aAOa2xldUhVjgOsmeP523VYuYU1g2Puwmik6PJk3a1XYu9fCX/Yk9LfYbiytjhWTb3b4qlqe1E0Aquczp0YWMxxX/tiA7YlHyY0tkYdbsEV9Dg6PfSeG16ADGLR17fmahvrO+vnPOrmLvjkBsMDh10rq95NEG6dCvT9LSjfGPe0NfbOdltzU93T1tpbjw9su3vamyaP3dwyhvpItTtH899uKAr72pAFv8g5XYT7ZCl3Eg8NK//hLXHw+PTJYp9tq1WA9HvbLnC2jLTa+ywAvdK/mkRnA08aB+3ZcVhyH7XcXTPoH7/izYVxeexSdwd9eLWfHbXaD3VvrnatKo9e6X7QjDEHxwkA4Rj3gP9bEvxLSgLrqAbP3w3/SYLLNFXCAn6Zix7j0JJfyi3VUtZcikEkXbGekMWG7YfRgrXzSfwMxobRiNKI+uHtq9YMhIFaXVlRhKTnegFA0wABjQPdHo9qmmGcU2LAHuJYNkU3iQGB0FGV5805dGRkNc+6OofCxPalU3TSNQ9VEr4KFXqlXTN9/iN0iYFA6liGDQxWuHXOPUQ1YCrtjXFW6bze0emBeXjlcGvMiYOCYF237Zq1bfTu1bc4WQ4bxmLCc2Hzol0kvByDsNzmAVZYzwjPsmhlILM3nTvCWynrWh5A50BZwYNNp3AjDvR22/cl0tHzuOeAOeBqHBzja3T5iTqZOmcuM+8YVJ7I2Q5hcdBrHhzCZrq+yi+WIJvXazyHiYc060ee/3riJH5bRwe4XAQMwJq3AqVpX/+nl5FEAlFflG1xDhuiLO6RUNsG927YIZCwLhawd+IeYu5JpZ5+J/aDKC2wj7kjOciV+6fdaoWFL4s5xtq4nu1+f7I1qWlIAGLMnr+QQZTUjTh+bhPXHEmhFfpSabEni6bems7cZUj807SWmq39k+ygzMOtm7OvDCoRpDPhlmpNQnKcS3+j46k7GC1XxVZyV0mXuEUSGQ7K/9isj7LfqI+ykaLwpNwQU4ZHFYuioT3N4gYaWI6iRPX8kLsve/RAmuojaqmrzBESNoZAEWvK7lzTv1jDbr7CGKwDow7VDo2BxuEtObsL73XoEMF+v4xphpEiv7Wn77czY8btZaip5HvT+ihmr19yJVrjtWXO/ViXs++XZ1vNukosaBYOkF54Wyz8jmW9wQEq9nG9wYOfw+IDPT7t3RobMpZ9KtnVkcFoovl0fHZqDObIALTvpOC8KlouvIcgejseXCYgp3su3BdoRYgvaGAFKbz/Wt0KjPduBBnXFsv4J5odpTH4x0WG3hiLDqVRs+cvW64qbiWJZYCTZ0/ICsKYoOMzcw4Sc1n4RoSw3gQQ40zSO7u3DUx75s0+DrDaHYFfoofECWw0+xs4ksY61zxGAinpY6F8lBgywuuNw2fSrxhfX2pSubEH7hH0YDBfF2Cevv3h3ZvihjWJGxOOM5mDQi67PEfHxBJ9rHNyYVNsCTo9s8dDNFvXTJeNslMY4KPs1Az53W0x9DRT4FLjeHO9zDhcaMwf7JcEH8/f8K8OQ4Q1eAW6R2B2PbaC77mVaRRY9uA1fqywZdOT0Sz6j2gUiHy9hv6up+NhhP87OYN/ZkEAZ0cBIBgawKK+wi3aywwjZJttvUbPz7UgZM5JMDHoMsSP+tgkxlArYA8RnLIBG4zlVhttMeF/u63XCfYM/eDxsKbBeOZ24FmDyTUIv902jR4YU2Jy0DvWvsUArGOYqNptcmrIWjoMQZ65B69UW2QUxbXOWxlVRaE9PCgsuWLl6qJrJ/TBRrg6LY4zaAT4P4PH0SJ5NIyefD2MLkHX4QcJSLNHeMbclECqM9z5LOg7HHg8L4vW3KD64DfF9bau17yD1/i1Tdnd9OGeLee84huKrkvclzCl+ZIV6J9pLXKmv351yIAts8tE0j25TmW8yWLeic8AEyQNTWQPWF5Nn5ig8ixRczZFBaoBm5vTL1Y7fqyZtR9BXp2RQKFqFKBDZ/Q8OlNXGulKxjYAO5aOnCmH/IA3mtdtkvwcfYUiLJWF2xKeStUZIvqzg+jM0oB8sB0NNhFfRUYLjNKZ8FckFB+NtLR9+vr5i+fP3v+AAYw/0i6tPkryXsHGCPasitZAoUcjyY6y5OzJMFo1oOLzczDM1ep6orZsZvPR4yf3WHVG/yKewihx6nKkRDX+w6lhICmqGSVOXQN1Udcoceo2bFW2IO5EoEsSA5uCuDSZ1RlMCguZNS22qjp3Dc9r0MTo96IPGviMFoIZXDhaupfFCr/i5VtZCllWUFAhkGcEGM8sxjJLx2yK9hLWNnmcWlo6Q7l85BI6L7hPwcDHJKITxIKhMahD3bl8IPS1Ny0PqAMLCKyie4JICMmvorPU2eS0l2hjJQ6t/jDhiKbRf0buu99PsH+w4mlJJT5A5bHFH/ZrTkbzfdbuNkl6F/cg9jnBjuW3fYDf/JJAtAdZ5wBAjo0DkRfeH6TtKyT4Qt2pDoR4msIsPXhAE3IE+8wCqLin6L6e4eZnQNeQv88sHjqbgNnYWHXr3aZqyXjZVct67ZxxcWt2V7Ufd4z9wpLT1NbSLoMqpW0IJz2UTIPJ6Gvzbd3Cj6HJs1uU1iQDBG7orELrKUHTEK0Cv8te+TCv2XJZzkvGVzZVBAEV/VF180dV2IvRul4BY65AVO9I0nLSQmkipNQ5fg+kO0pxEYJJriMFkVO4L05r5s8SLfLLDN9HYyHpvX0gjQy5xqb4OMtwI5sAWc/uR1YxmbCsF+VmggBAPT5KUymYjKXHO7IjkTQ5pJgwvpPhdkndEK3scZpr2ACC7L2ZnKXWKMXCVCM8MSZUo24U8q9wUACWc0sFrfBNwlGyBAQ/PHYbcPQsvKc27OkpKBinCLYsXlmgCDZRs4CjSyGzmPPPuzQzZiTRc14DGdtF3IdiLBR3TfDlovvSqycExd4PDUJRgfb5C4mUxLVEhp5t4hwE8Q3Veck/kHHOfop1uaqAGUFmNWLj4FSA/QFUKtqJfRhl2LasqNT8Sgl/mn39DWzI4c/jJ/Tn9MlM2qNydBn9TBUTPqKVZjJdtwgAHo3+GyGORl/zP48/A7CeBRXnANtKHAmt0m7hM49adfzhHpyjNzIcO9jgdeVqV+/aBPSFc/pHeyj1mWXMY/QoLJUvMfeUD4+PRN6TWHg+EgdnWFFuqwtY/nc2O5uFOhvd0WoU7uyuZo/MZntrk/KnH17evS+5xwbiYrt0t9ePTm3F/VhvqkcHNtUAKeOiFNTwx10JA8lRPeHqJjOlzzjBQ7f2s3bC0CX/FrKV+98NRhvKTDM5JeWgL64BnKLaWOTUcTwSj08tj8Qw+oU1NacaVCgXu2JteHIOeShOfRfFfylrZ6Sp+cinpudrcJwS8hhQVLNY4yXwTtWWc+70geF8eT6RHOi85IEtm7IiB0FwBoINiutjG9Aoy2VJMZ0wrh+BwxhJ40Q7ZEanX38zlOlIoPjVizc/JfxR1MB3YIGkvZaysPWO5kaSOFILKjIkijsFPNxgguBJjeQBKHOchsV1sOGZ2ZCkznEdjrwOR8d1+MhsuBRcheqBj/Yrif1XAhtRMEJ18XV2GlyqegITCVGu2W3DcNOBUW3nRUvhOf5ZBGEsE0E09XmBOry7ERIUHTEL/BwbFMQwuqBlrH1fNiw+edxEgEHRORZ0mRrFMmVNX1SKjKaj6BB+uOVHh2FE9lJG2Yxvg0cJezesKsZonk8sp+EIhUfPPfX4UEVF/sMLfaFAck0ypXWMMreNmBLQSDIGEOEbDdAngC46NyiG8qSM+WyZKswLeuGEBg74o8i0VpOygHVvsESOX+Qm1gH82ArlGjqfoYnMIbxUfTVsli7Yp3Ku9sL8l0yyBgtAu+KG+pN1wAaMW5AgC86APFjGxEN+wUpf3+HRD2EugjCIFlFOWYuAhHMMPproWtm8YXhqy9ckbkh0djXNgfFfyvcn3548GsUER+AzcfAbivFN+B+D7PyracweKEQrfhtuubMfn41QNIrzPJ49R+9gYWo0QWE5Ftt8Xc8p9GwSz7c7QEtoz7yu1jemTqOP0+ns0cha1J/iaQhGEn63nnKVOe8MrSuQJohdnfBRpuaw5NcXd4do8Pg+bkrDw4Z1hfj4mHv1yLOn1cR6114kTu4w49xSwAl98GpH87nbP9EwNcajGtB5othc7jogd9uJOjnZ+Pw5w+eAfxvdzqyiuCWzJvrp4/luUbjpuuTZBdKOcwvpYnFUzJFP+1qox4fqEdZ0w/fO6I65ZGyLzzSZqA42W9JcZ+zkbGSDXderkpwwOM+JBGcGW0k5VJrkrJcdajTeWvoj+Y4ftvRck2bAqmDlUJ7IxI5xSET8N4lbqURSS9ug8wVDQyWvDG1UDkSDezqOs+KU+pw5aRsQKIutnl20NOqCfzJMTmCaTRJH9UIf8og4dx7ToYPcKfJVIKaiW9+8/iHiaEUcLfM7dx6qMAkEFlBQrZbLPrqaLkIkcgZLAwMQVeX8cD+jIAf/4RFFfFol1xZqK0O82zzMF/bAfcYsrkJ0IxsOrRReqznY/Wva7aroBc6FuLAwfqH4VJRrXHtmEINRhW223Y0I4LMUox+p/HkBendoVjw2/Ecp1h71yTPnTqxz67+FPuLC7XiFFFJEElcOK6yKlBpHtJVKh13ihrxkiivs2DGjJlABiJag6TXimRazs8eu08OEzNCH9gx4AOv313tf821M0l/lLcZLr7H3Hu9V0PdkwJtZJt+/dfJn6mSlConRPCX8r6gJrcRB/xRKEATSiZRIX1j/qSX3T6z+/rUUH6aM+ntovh4l14rgEuFM82JObPUSdOt9KZXoaLO/g0YUwfNGwJFYf7TqDigHuRKRHtojJSkS/7olSembcCu8zYqWfvDFKMU3lMt4IWB+lJawyRk9fmx4urjfh683NSP02RaXrzxDVLZlzWYHQgY22nRGZp1+uMfa8hSFfzgm+SYxu/on0UoCYevs3+CAkLpCenO9dJzKOtUqywQnVQl+4OLoFVcBOZ4qSixXN+WqxFgWyVLKD+2cDJpKKBD0wKVx4AWXzKGDQo2udDaq2TYOy4J60xPQkjf+H4pqSqsnYmbFXQnJkqSl1UmvG/cm7xoKc6BGU/HJ8gxFCF95nNdVitEb2lzr+p5z1GvK8/WpxvycVIARTlAQGzwDmOylw88xt/B/+NttRXpM846GhOM9FCCHPMtRO+F8DvQqPrG1TXMjeyQei3IXtHkmSiNKrSSPuw3PqcArJZJYgHUqIq2c2vQNt1f7LFC7q0S9rjKLl1tRvNxaxbL20qrdydqdVVvecqE81fa1Fw7xLLS8mzIUjJ47NA4BE0kj8JYMBca8NMObRdjXcYf55FQcHy5KfivG5NSCvORnXp8D+ewIyOq6D8PXb10A8ivhL88UYHnPyOdDFJeOaDqYl5BYYOtzq6F3ZYrml/BlKoeAuSsUYE2nkr8lR+P+U3Kx5NvZzE4SKi7oEf78RH/zrw6vpBxvHcMzEgIQCNUYH9uExJ5Ix2TAmprPUgzSUhfGFs98dEwL8UkEpm3GW4Im0a19rpPzvM0PHriC2wAoEr/rT8KNsfHvbDSofS66Et+Cy57Fbk2nbPDNx2MTfczri7oRn8HCTINRnEyRhjw90ZT+MZJUEBl0hgmY9nJVAQNxDeZ8z2HjF0z/4af+CFF08LlpM75AyoyYz0/OCeXnFcm3sDfMD+fp8RmCQ3O8THvHGtIdycuSnHQNNoF7GqkkBYf4zG+FyQ6cVAZiuSAdAxwn7vwqK1qXZONfsps28emt8rUZxpUAKxZjiPfkDBIL8ufZUWysAB/mVmNgfxtW1SzIH47hNQP7vyGjiV56uMwgTKj6UfzlNOljLh1JPQ0YPQetmR7rpMe06LcLAho9oJIPKtuh8bnuppg3tZVY7yDXPHhwu4ypjbEkbzlZ9kqLe6t1ymvMMoxNhHXFrymiMrp+gpN1H+xHTI3XiT1l9+4hvqob2OdoVPsNUH84gfmFnsvKtrTP2RfuALbZsoN7aWlOSNgJbIrmRupqKuzbqAhFjkmt+JOVqR0aUoZ2+GuUe6Ow0lWJMot9zQlUYseuKY0z+wbFnkzMB6+S0jYYwCt26671mlvLwLisAwSdlWjEXF5G7g6znriE8aGTxYRumuQGZhsPD10rQJGMB9L+O+/9lPxOBTe/vvM6mFbf7cPL8uLm9glncgayfH/FqvZNcf4QU6bsKtivS+doe/Lik6UrnPzO0BjP3dxKTs7mcfT1I/NtID3zODo7PT21KOJkUx1Hj09P7TqBXJ4Ex5XmRvrScTQ69Qkrffjw+tHImdfgW5UrVLC6kZaKc6+6nY3ep0ZSIYyNE5VEXnS8mjNDQiUH815zINU2u2cL7nkCDtthprIjG93bqWXAzzGhyR19GFGBwct4zIU7k9kd1DaIBKmXcN2oIDJtGWLXqy0lm4ANkrRCmd4HWr6PlckKFck9SFZr4mBn7TqHDnwzjZkEcnTadsoOxaOre+5EsrOUDe0k9QcyX8mERwTLyXskri+SXzSgY8zbF+LnP3dn+/2KGisCBJLG8iNuB1hPHmLnch26aUvi6ebq/QtuHOTlAyoVyUXReulA+Hm5Gu+tfNpn+uaE/rwy2WKH963gDcBJmuHtcekBVJzm8poHEMMfd5SqKpg0zuHPnnRc+a7Fb0F8e9k+dQMeWtpmJt1xdRsLTaMuVRtSTlk/LRu93FuH3XSDnUwaQxwVStSM1+HBXklcqmdoSHE1pHPpUNC4ctYuUkPBaaU95QzWzaznkZOyp4m2t/LGr0TfoORfQMMdS/Sdft89fgKsvMNPRcvjcY463uI/E35AdFAO887oCNLOf/NcnJzeclj73+mMZcK/7WXAMa+S0nGv1vd/kgvA/g3YIzNrXg/U8+ZTZt8Haw32zUUZD8yMPiRVYTNgm39+WN84klep+kHObrapYSjbDV6fZh9OGWMwONPx8pgRWYoAhtxyLQz3yyU30rj3WikVu+GPVAWd/aqBOkbsPcfpmErmML3LqQ6cD/qD04EF3ujMEclRuKb2TMZuu+jsB55VcX6Ty+P2W3nDZouGWuANrm7Tbas4leSdYlu52MeBE0qV79ExbLwdolFBpxHVt+2a4Pou91Of+zk5C53mvyJroRMkozqk6+5sHS0u6aLT+5x9BLsxcOGFbB/Qr+ooMR0eUsO62l03ZBgEpnRboutAPS5g3wFb4FU2kcyjCjaCMWEmvMy5HsW/Z8RHQLGPg7lTU9oCzhwGVH244+M6/ewOQytrqqk0sznf+LrJXnJOk95zIOsIyDK/peGsnLshzJRbOeTmlQMPjk0YHXjIyF0H3nq1CDQw85otmBm9yaNgDokXmistYOwReKIG4U89X6p7Mu8cXx08u08PwnY/b/JAexVcyAjV57I7aJxTqz6vMLYxr/LRbKAjhTnPTQNOspm4OwbfU44IhzsV24SOB4ypRizEGcdhFBy/25fsX40u7HQ3qXvUiY49rruB3u3G516SDWyCy5ZHH5lHCT2R3Z95tPLgQfjmgKV0r+Z07W0sruUO1/VMfDmN0kU783Y9jq6krL7eZoLu3/PzuIdGIXvkjt9Z31lNSKgcYCsXazPuXM/Q3XMu3do9ky1e28a97bTFbYfpigh5TXXUTk+ycl3VDNzRtoN5sB+IylGtzUCeYOtgTE/RrJhx1kB4wgQHPChD56AHq956mTZk4MOtcRLgMd5QnwU4LPLFmGJ/zGlHrNnFPeQQP83Nq648nYplLAdy8LAmUEmets3sJHai08H/AYt61f6NigAA"""
AIDE_RUNTIME_GZIP_BASE64 = """..."""

runtime_dir = Path("/kaggle/working/tiny_jpeg_bias_runtime")
runtime_dir.mkdir(parents=True, exist_ok=True)
runner_path = runtime_dir / "test_tiny_jpeg_bias_three_models_kaggle.py"
common_path = runtime_dir / "test_three_models_commfor_unseen_kaggle.py"
aide_path = runtime_dir / "train_aide_original_full_tiny_commfor_kaggle.py"

runner_path.write_bytes(gzip.decompress(base64.b64decode(RUNNER_GZIP_BASE64)))
common_path.write_bytes(gzip.decompress(base64.b64decode(COMMON_RUNTIME_GZIP_BASE64)))
aide_path.write_bytes(gzip.decompress(base64.b64decode(AIDE_RUNTIME_GZIP_BASE64)))

sys.path.insert(0, str(runtime_dir))
spec = importlib.util.spec_from_file_location("tiny_jpeg_bias_runner", runner_path)
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)

CONFIG["common_runtime_path"] = str(common_path)
CONFIG["aide_runtime_path"] = str(aide_path)
print("Runner:", runner_path)
print("Common runtime:", common_path)
print("Full AIDE runtime:", aide_path)

In [ ]:
result = runner.run_evaluation(CONFIG)
result

In [ ]:
import json
import pandas as pd
from IPython.display import Image as DisplayImage, display

run_dir = Path(CONFIG["output_root"])

print("Dataset summary")
display(pd.DataFrame([json.loads((run_dir / "dataset/overall_statistics.json").read_text())]))

print("Generator / label counts")
display(pd.read_csv(run_dir / "dataset/generator_label_counts.csv"))

print("Native formats")
display(pd.read_csv(run_dir / "dataset/format_distribution.csv"))

print("Case sample counts")
display(pd.read_csv(run_dir / "dataset/case_sample_counts.csv"))

print("Top native image sizes")
sizes = pd.read_csv(run_dir / "dataset/exact_size_distribution.csv")
display(sizes.groupby(["native_width", "native_height", "native_size"], as_index=False)["count"].sum().sort_values("count", ascending=False).head(30))

for plot_name in [
    "format_by_label.png",
    "top_native_sizes.png",
    "size_bins_by_label.png",
    "aspect_ratio_by_label.png",
    "jpeg_quality_by_label.png",
]:
    plot_path = run_dir / "plots" / plot_name
    if plot_path.is_file():
        print(plot_name)
        display(DisplayImage(filename=str(plot_path)))

In [ ]:
print("Overall metrics (%)")
overall = pd.read_csv(run_dir / "metrics/overall_by_model_case.csv")
metric_columns = [
    "accuracy", "balanced_accuracy", "real_recall", "fake_recall",
    "fake_precision", "fake_f1", "roc_auc", "average_precision",
]
shown = overall.copy()
for column in metric_columns:
    shown[column] = (100 * shown[column]).round(2)
display(shown[["model", "case", "num_samples", *metric_columns]])

print("Delta versus matching raw cohort — percentage points")
delta = pd.read_csv(run_dir / "metrics/delta_from_matching_raw.csv")
delta_columns = [column for column in delta.columns if column.startswith("delta_")]
shown_delta = delta.copy()
for column in delta_columns:
    shown_delta[column] = (100 * shown_delta[column]).round(2)
display(shown_delta)

print("Per-generator metrics")
display(pd.read_csv(run_dir / "metrics/generator_by_model_case.csv"))

In [ ]:
import shutil
from IPython.display import FileLink, display

archive_base = "/kaggle/working/tiny_jpeg_bias_three_models_results"
archive_path = shutil.make_archive(archive_base, "zip", root_dir=run_dir)
print("Download complete result package:")
display(FileLink(archive_path))

## Output chính

- `dataset/image_audit.csv`: audit từng ảnh test.
- `dataset/exact_size_distribution.csv`: toàn bộ kích thước `W×H`.
- `dataset/controlled_jpeg96_manifest.csv`: membership của controlled cohort.
- `predictions/<model>/<case>.csv`: prediction từng model/case.
- `metrics/overall_by_model_case.csv`: metric tổng quan.
- `metrics/generator_by_model_case.csv`: metric từng generator.
- `metrics/delta_from_matching_raw.csv`: mức thay đổi so với raw tương ứng.
- `metrics/paired_probability_shift.csv`: thay đổi xác suất trên cùng sample.
- `tiny_jpeg_bias_three_models_results.zip`: gói tải về.

`all_jpeg96` có thể tạo double-compression trên real JPEG; kết luận bias chính nên dựa vào `fake_jpeg96` và cặp `raw_on_controlled_cohort ↔ controlled_jpeg96`.